In [ ]:
import re
from pathlib import Path

import pandas as pd

# ============================================================
# 1. Paths (assuming notebook lives in: project/notebooks/*.ipynb)
# ============================================================

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data"

# You can ignore product_names if you want, leaving it here for completeness
PRODUCT_NAMES_FILE = DATA_RAW / "insights" / "product_names.csv"
PRODUCT_NAMES_NORM_FILE = DATA_RAW / "insights" / "product_names.csv"   # in-place (if used)

PAK_FILE = DATA_RAW / "raw" / "pakistani_amazon.csv"
PAK_OUT_FILE = DATA_RAW / "raw" / "pakistani_amazon.csv"           # in-place

print("Project root:        ", PROJECT_ROOT)
print("Pakistani file:      ", PAK_FILE)
print()


# ============================================================
# 2. Brand normalization mapping
#    We will use this ONLY to replace text directly in the CSV.
# ============================================================

# Brand normalization mapping (keys: raw patterns you expect to see)
BRAND_NORMALIZATION = {
    # Case unification
    "mi": "MI",
    "philips": "Philips",
    "tp-link": "TP-Link",
    "amazonbasics": "AmazonBasics",
    "amazon basics": "AmazonBasics",
    "ikea": "IKEA",

    # Example: if you later want families:
    # "redmi": "Xiaomi",
    # "poco": "Xiaomi",
    # "xiaomi": "Xiaomi",
}


# ============================================================
# 3. (OPTIONAL) You can skip this if product_names.csv is no longer needed
# ============================================================

if PRODUCT_NAMES_FILE.exists():
    print("=== (Optional) product_names.csv EXISTS, but not modifying structure ===")
    # If you really don't care, you can comment this whole block out.
    pn = pd.read_csv(PRODUCT_NAMES_FILE)
    # Simple direct replacement in product_names too, if you want consistency:
    if "product_name" in pn.columns:
        for raw, canonical in BRAND_NORMALIZATION.items():
            pattern = re.compile(re.escape(raw), flags=re.IGNORECASE)
            mask = pn["product_name"].astype(str).str.contains(pattern, na=False)
            if mask.any():
                print(f"[product_names] Replacing '{raw}' -> '{canonical}' in {mask.sum()} rows")
                pn.loc[mask, "product_name"] = (
                    pn.loc[mask, "product_name"]
                    .astype(str)
                    .str.replace(pattern, canonical, regex=True)
                )
        pn.to_csv(PRODUCT_NAMES_NORM_FILE, index=False)
        print("Saved updated product_names.csv")
    print()


# ============================================================
# 4. Normalize TEXT INSIDE pakistani_amazon.csv USING MAPPING
#    - No brand_guess
#    - For each mapping (raw -> canonical), search all string columns
#      and replace occurrences (case-insensitive) in-place.
# ============================================================

print("=== Normalizing text inside pakistani_amazon.csv ===")
if not PAK_FILE.exists():
    raise FileNotFoundError(f"Cannot find {PAK_FILE}")

df = pd.read_csv(PAK_FILE)

# We only touch object (string-like) columns
# obj_cols = df.select_dtypes(include=["object"]).columns.tolist()
obj_cols = ["product_name"]
print("String columns that will be normalized:", obj_cols)
print()

for raw, canonical in BRAND_NORMALIZATION.items():
    raw = raw.strip()
    if not raw:
        continue

    # Match whole word only: \braw\b
    # So "mi" matches "mi", "MI," "mi." etc.
    # but NOT "premium"
    pattern = re.compile(rf"\b{re.escape(raw)}\b", flags=re.IGNORECASE)

    total_hits = 0
    for col in obj_cols:
        s = df[col].astype(str)
        mask = s.str.contains(pattern, na=False)
        hits = mask.sum()
        if hits:
            total_hits += hits
            print(f"Column '{col}': replacing whole word '{raw}' -> '{canonical}' in {hits} cell(s)")
            df.loc[mask, col] = (
                s[mask].str.replace(pattern, canonical, regex=True)
            )

    if total_hits == 0:
        print(f"No whole-word occurrences of '{raw}' found in any string column.")
    print()


# Save back in-place
df.to_csv(PAK_OUT_FILE, index=False)
print("Saved normalized Pakistani dataset to:", PAK_OUT_FILE)
print("=== Done. All replacements applied directly to the CSV ===")


Project root:         d:\Users\DELL\new-onedrive\OneDrive - Institute of Business Administration\Documents\sem7\mlops\project
Pakistani file:       d:\Users\DELL\new-onedrive\OneDrive - Institute of Business Administration\Documents\sem7\mlops\project\data\raw\pakistani_amazon.csv

=== (Optional) product_names.csv EXISTS, but not modifying structure ===
[product_names] Replacing 'mi' -> 'MI' in 242 rows
[product_names] Replacing 'philips' -> 'Philips' in 22 rows
[product_names] Replacing 'tp-link' -> 'TP-Link' in 18 rows
[product_names] Replacing 'amazonbasics' -> 'AmazonBasics' in 29 rows
Saved updated product_names.csv

=== Normalizing text inside pakistani_amazon.csv ===
String columns that will be normalized: ['product_name']

Column 'product_name': replacing whole word 'mi' -> 'MI' in 31 cell(s)

Column 'product_name': replacing whole word 'philips' -> 'Philips' in 22 cell(s)

Column 'product_name': replacing whole word 'tp-link' -> 'TP-Link' in 20 cell(s)

Column 'product_name': 